In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

# 1. تثبيت البذرة العشوائية لضمان قابلية إعادة التشغيل (Reproducibility)
SEED = 42
np.random.seed(SEED)

# 2. تحميل البيانات (تم تعديل المسار الصحيح للملف المتاح)
DATA_PATH = "/content/diabetes_012_health_indicators_BRFSS2015.csv"
df = pd.read_csv(DATA_PATH)

print("شكل البيانات (عدد الصفوف، عدد الأعمدة):", df.shape)
print("\nأول 5 صفوف:")
print(df.head())

# 3. التحليل الاستكشافي السريع (EDA)
target_col = "Diabetes_binary" if "Diabetes_binary" in df.columns else "Diabetes_012"
print("\nالقيم المفقودة في كل عمود:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nعدد الصفوف المكررة:", df.duplicated().sum())
print("\nتوزيع الفئات (الهدف):")
print(df[target_col].value_counts(normalize=True) * 100)

# رسم توزيع الفئة الهدف وحفظه
plt.figure(figsize=(6, 4))
df[target_col].value_counts().plot(kind="bar")
plt.title("Target Class Distribution")
plt.xlabel(target_col)
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("target_distribution.png", dpi=300)
plt.close()

# إزالة التكرارات لتنظيف أولي (اختياري وموثق ضمن مرحلة التنظيف)
df_clean = df.drop_duplicates().copy()

# تحويل الهدف إلى ثنائي إذا كان 3 فئات (0 = سليم، 1 = ما قبل السكري أو سكري)
if target_col == "Diabetes_012":
    y = (df_clean[target_col] > 0).astype(int)
else:
    y = df_clean[target_col].astype(int)

X = df_clean.drop(columns=[target_col])

# 4. تقسيم البيانات إلى تدريب واختبار قبل أي معالجة لتجنب تسرب البيانات (Data Leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

# تحديد الأعمدة الرقمية المستمرة مقابل الفئوية/الترتيبية
continuous_cols = [c for c in ["BMI", "MentHlth", "PhysHlth"] if c in X.columns]
other_cols = [c for c in X.columns if c not in continuous_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("scaler", StandardScaler(), continuous_cols),
        ("passthrough", "passthrough", other_cols),
    ]
)

# 5. تعريف النموذج المرجعي (Baseline) وثلاث خوارزميات تصنيف مع مراعاة عدم توازن الفئات
models = {
    "Baseline (Most Frequent)": DummyClassifier(
        strategy="most_frequent", random_state=SEED
    ),
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=SEED
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=8, class_weight="balanced", random_state=SEED
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=150,
        max_depth=12,
        class_weight="balanced",
        n_jobs=-1,
        random_state=SEED,
    ),
}

# 6. تقييم النماذج عبر التحقق المتقاطع الطبقي (Stratified K-Fold)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
}

print("\n=== نتائج التحقق المتقاطع (5-Fold CV على بيانات التدريب) ===")
for name, clf in models.items():
    pipe = Pipeline(steps=[("prep", preprocessor), ("model", clf)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    print(
        f"{name:26s} | "
        f"Recall: {scores['test_recall'].mean():.3f} | "
        f"Precision: {scores['test_precision'].mean():.3f} | "
        f"F1: {scores['test_f1'].mean():.3f} | "
        f"ROC-AUC: {scores['test_roc_auc'].mean():.3f}"
    )

# 7. تدريب أفضل نموذج (مثلاً Random Forest) وتقييمه النهائي على مجموعة الاختبار
best_pipe = Pipeline(
    steps=[
        ("prep", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=150,
                max_depth=12,
                class_weight="balanced",
                n_jobs=-1,
                random_state=SEED,
            ),
        ),
    ]
)
best_pipe.fit(X_train, y_train)

y_pred = best_pipe.predict(X_test)
y_prob = best_pipe.predict_proba(X_test)[:, 1]

print("\n=== التقييم النهائي على مجموعة الاختبار (Test Set) ===")
print(
    f"Accuracy : {best_pipe.score(X_test, y_test):.4f}\n"
    f"Precision: {precision_score(y_test, y_pred):.4f}\n"
    f"Recall   : {recall_score(y_test, y_pred):.4f}\n"
    f"F1-Score : {f1_score(y_test, y_pred):.4f}\n"
    f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}"
)
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

شكل البيانات (عدد الصفوف، عدد الأعمدة): (253680, 22)

أول 5 صفوف:
   Diabetes_012  HighBP  HighChol  CholCheck   BMI  Smoker  Stroke  \
0           0.0     1.0       1.0        1.0  40.0     1.0     0.0   
1           0.0     0.0       0.0        0.0  25.0     1.0     0.0   
2           0.0     1.0       1.0        1.0  28.0     0.0     0.0   
3           0.0     1.0       0.0        1.0  27.0     0.0     0.0   
4           0.0     1.0       1.0        1.0  24.0     0.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  ...  AnyHealthcare  \
0                   0.0           0.0     0.0  ...            1.0   
1                   0.0           1.0     0.0  ...            0.0   
2                   0.0           0.0     1.0  ...            1.0   
3                   0.0           1.0     1.0  ...            1.0   
4                   0.0           1.0     1.0  ...            1.0   

   NoDocbcCost  GenHlth  MentHlth  PhysHlth  DiffWalk  Sex   Age  Education  \
0          0.0     

In [4]:
# عدد الصفوف (Rows) وعدد الأعمدة (Columns)
rows, cols = df.shape
print(f"عدد الصفوف (Rows): {rows}")
print(f"عدد الأعمدة (Columns): {cols}")

# أسماء جميع الأعمدة
print("\nأسماء الأعمدة:")
print(df.columns.tolist())

# عرض صف معين حسب رقمه (مثلاً الصف رقم 0)
print("\nالصف الأول (Row 0):")
print(df.iloc[0])

# عرض عمود معين حسب اسمه (مثلاً عمود BMI)
print("\nأول 5 قيم من عمود BMI:")
print(df["BMI"].head())

عدد الصفوف (Rows): 253680
عدد الأعمدة (Columns): 22

أسماء الأعمدة:
['Diabetes_012', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']

الصف الأول (Row 0):
Diabetes_012             0.0
HighBP                   1.0
HighChol                 1.0
CholCheck                1.0
BMI                     40.0
Smoker                   1.0
Stroke                   0.0
HeartDiseaseorAttack     0.0
PhysActivity             0.0
Fruits                   0.0
Veggies                  1.0
HvyAlcoholConsump        0.0
AnyHealthcare            1.0
NoDocbcCost              0.0
GenHlth                  5.0
MentHlth                18.0
PhysHlth                15.0
DiffWalk                 1.0
Sex                      0.0
Age                      9.0
Education                4.0
Income                   